# Multi-Label ECG Interpretation & Structured Clinical Report Generation — PTB-XL

A five-stage pipeline that goes from raw 12-lead ECG waveforms to a one-page clinical PDF report — **without ever letting an LLM invent a measurement or a diagnosis**:

```
 raw waveform ──► [Stage 1] 1D ResNet, 5 sigmoid outputs ──► class probabilities (NORM/MI/STTC/CD/HYP)
 raw waveform ──► [Stage 2] signal processing ─────────────► HR, rhythm, PR/QRS/QT/QTc, P/QRS/T axes
 PTB-XL meta  ──► [Stage 3] structured case JSON ◄─ merge ─┘  (age, sex, readable diagnoses)
                      │
                      ▼
                [Stage 4] LLM phrasing ONLY (input = the JSON, never the waveform)
                      │        └─ guarded: numbers in the output must exist in the input
                      ▼
                [Stage 5] fpdf2 ──► one-page PDF per case
```

**Hard constraint (enforced, not just promised):** the LLM receives *only* the structured JSON from Stage 3. A validation step extracts every number from the generated narrative and checks it against the numbers present in the input JSON — any value not present is flagged as a violation.

**Dataset:** [PTB-XL](https://physionet.org/content/ptb-xl/) — 21,837 records, 18,885 patients, 12-lead, 10 s, 500 Hz → tensor shape `(12, 5000)`.

**Kaggle setup**
1. **Settings → Accelerator → GPU T4 x2** (a single T4 is enough; ~35–45 min total)
2. **+ Add Input** → your PTB-XL dataset (auto-discovered anywhere under /kaggle/input)
3. *Optional:* **Add-ons → Secrets → GEMINI_API_KEY** so Stage 4 uses a real LLM. Without a key the notebook still runs end-to-end using a deterministic template phraser.
4. Run all cells.

**Pipeline stages in this notebook**

| Stage | What it does | Where |
|---|---|---|
| 1 | Multi-label 1D ResNet classifier (BCEWithLogitsLoss, official folds 1–8/9/10) | Cells 7–17 |
| 2 | Signal measurement extraction (HR, PR/QRS/QT/QTc, axes, rhythm) | Cells 18–20 |
| 3 | Structured case JSON (predictions + measurements + metadata) | Cells 21–22 |
| 4 | LLM narrative layer with numeric guardrail | Cells 23–24 |
| 5 | One-page PDF reports | Cells 25–26 |
| Eval | AUROC/AUPRC/F1/sens/spec · HR accuracy · finding P/R · narrative rubric | Cells 17, 27–29 |

In [11]:
!pip install -q wfdb fpdf2

## 1. Configuration & data acquisition

`DATA_DIR` is auto-discovered under `/kaggle/input/**` (any attached dataset containing `ptbxl_database.csv`). Override the first line manually if your dataset lives elsewhere. Falls back to the local repo layout (`data/raw/ptbxl`) when not on Kaggle.

In [ ]:
import ast
import json
import math
import os
import random
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Paths ──────────────────────────────────────────────────────
IN_KAGGLE = Path("/kaggle/input").exists()
WORK = Path("/kaggle/working") if IN_KAGGLE else Path("kaggle_nb_output")
WORK.mkdir(parents=True, exist_ok=True)

# Default: the attached PTB-XL upload; auto-discovered anywhere else under
# /kaggle/input if the default path is not present.
DATA_DIR = Path("/kaggle/input/datasets/khyeh0719/ptb-xl-dataset/"
                "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1")
if IN_KAGGLE and not (DATA_DIR / "ptbxl_database.csv").exists():
    for cand in Path("/kaggle/input").rglob("ptbxl_database.csv"):
        DATA_DIR = cand.parent
        print(f"Auto-discovered PTB-XL metadata at {DATA_DIR}")
        break

# ── Official PTB-XL folds (patient-wise, no leakage) ───────────
TRAIN_FOLDS = list(range(1, 9))   # folds 1–8
VAL_FOLD = 9
TEST_FOLD = 10

SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]  # fixed output order
FS = 500                                          # PTB-XL filename_hr rate

assert (DATA_DIR / "ptbxl_database.csv").exists(), (
    f"ptbxl_database.csv not found under {DATA_DIR} — attach the PTB-XL dataset "
    "or set DATA_DIR manually."
)
print(f"DATA_DIR = {DATA_DIR}")
print(f"DEVICE   = {DEVICE}")
print(f"WORK     = {WORK}")

## 2. Shared preprocessing pipeline

**SHARED MODULE** — identical to the one used at inference time in the CardioLens platform (`ml/preprocessing.py`). Changing anything here silently changes model behaviour downstream.

Steps: bandpass filter (0.5–45 Hz) → resample to 500 Hz → segment into 10 s windows → z-score normalize per lead.

In [13]:
from scipy.signal import butter, filtfilt, resample

TARGET_FS = 500
BANDPASS_LOW = 0.5
BANDPASS_HIGH = 45.0
SEGMENT_SEC = 10
NUM_LEADS = 12


def bandpass_filter(signal, fs, low=BANDPASS_LOW, high=BANDPASS_HIGH, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    return np.apply_along_axis(lambda x: filtfilt(b, a, x), axis=0, arr=signal)


def resample_signal(signal, fs_orig, fs_target):
    n_target = int(len(signal) * fs_target / fs_orig)
    return resample(signal, n_target, axis=0)


def segment_signal(signal, fs, seg_sec=SEGMENT_SEC):
    seg_samples = fs * seg_sec
    n = len(signal)
    n_segs = n // seg_samples
    if n_segs == 0:
        padded = np.zeros((seg_samples, signal.shape[1]))
        padded[:n] = signal
        return padded.reshape(1, seg_samples, signal.shape[1])
    trimmed = signal[: n_segs * seg_samples]
    return trimmed.reshape(n_segs, seg_samples, signal.shape[1])


def normalize_segments(segments):
    mean = segments.mean(axis=1, keepdims=True)
    std = segments.std(axis=1, keepdims=True)
    std = np.where(std == 0, 1.0, std)
    return (segments - mean) / std


def preprocess_ecg(signal, fs):
    # Full pipeline: filter -> resample -> segment -> normalize. (segments, 5000, 12)
    if signal.ndim == 1:
        signal = signal.reshape(-1, 1)
    filtered = bandpass_filter(signal, fs)
    if fs != TARGET_FS:
        filtered = resample_signal(filtered, fs, TARGET_FS)
    segments = segment_signal(filtered, TARGET_FS)
    return normalize_segments(segments)


print("Preprocessing pipeline defined (shared with inference).")

Preprocessing pipeline defined (shared with inference).


## 3. Stage 1a — Multi-label ground truth & official folds

Each record's `scp_codes` are mapped to the **set** of diagnostic superclasses it carries (via `scp_statements.csv`, the authoritative mapping shipped with PTB-XL) — no priority collapse to a single class. A record can therefore be e.g. `MI + STTC` simultaneously.

Splits use the **official `strat_fold` column** (patient-wise by construction): train = folds 1–8, val = fold 9, test = fold 10. The cell verifies patient disjointness to demonstrate there is no leakage.

`pos_weight` (negative/positive ratio per class, from the training folds only) compensates the class imbalance inside `BCEWithLogitsLoss`.

In [ ]:
scp = pd.read_csv(DATA_DIR / "scp_statements.csv", index_col=0)
diagnostic = scp[scp.diagnostic == 1]
CODE_TO_SUPERCLASS = dict(zip(diagnostic.index, diagnostic.diagnostic_class))
SCP_RHYTHM_CODES = set(scp[scp.rhythm == 1].index)

db = pd.read_csv(DATA_DIR / "ptbxl_database.csv", index_col="ecg_id")
db["scp_codes"] = db["scp_codes"].apply(ast.literal_eval)
SEX_MAP = {0: "male", 1: "female"}


def _finite_int(v):
    # int for usable numeric values, else None. Series.get() yields None for a
    # column absent from this PTB-XL version; np.isfinite handles NaN entries.
    if v is None:
        return None
    try:
        return int(v) if np.isfinite(v) else None
    except (TypeError, ValueError):
        return None


# Older PTB-XL releases (1.0.0/1.0.1) ship no hr / heart_axis / report columns
# - those fields degrade to "unavailable" instead of crashing the pipeline.
MISSING_META = [c for c in ("hr", "heart_axis", "report") if c not in db.columns]
if MISSING_META:
    print(f"Note: {MISSING_META} missing from ptbxl_database.csv (older PTB-XL "
          f"dataset version) - those fields will be reported as unavailable")

RECORDS = []   # one dict per usable record
skipped = 0
for ecg_id, row in db.iterrows():
    supers = sorted({CODE_TO_SUPERCLASS[c] for c in row["scp_codes"] if c in CODE_TO_SUPERCLASS})
    if not supers:  # records with no diagnostic superclass (e.g. pacing-only)
        skipped += 1
        continue
    rid = str(ecg_id).zfill(5)
    axis_v = row.get("heart_axis")
    report_v = row.get("report")
    RECORDS.append({
        "rid": rid,
        "fold": int(row["strat_fold"]),
        "supers": supers,
        "y": np.array([1 if s in supers else 0 for s in SUPERCLASSES], dtype=np.uint8),
        "filename_hr": row["filename_hr"],
        "age": _finite_int(row.get("age")),
        "sex": SEX_MAP.get(_finite_int(row.get("sex"))),
        "hr": _finite_int(row.get("hr")),
        "heart_axis": axis_v if isinstance(axis_v, str) else None,
        "report": report_v if isinstance(report_v, str) else "",
        "scp_codes": dict(row["scp_codes"]),
    })

REC_BY_ID = {r["rid"]: r for r in RECORDS}
REC_INDEX = {r["rid"]: i for i, r in enumerate(RECORDS)}
y_all = np.stack([r["y"] for r in RECORDS]).astype(np.float32)

train_idx = np.array([i for i, r in enumerate(RECORDS) if r["fold"] in TRAIN_FOLDS])
val_idx = np.array([i for i, r in enumerate(RECORDS) if r["fold"] == VAL_FOLD])
test_idx = np.array([i for i, r in enumerate(RECORDS) if r["fold"] == TEST_FOLD])

# ── Leak check: folds must not share patients ──────────────────
patient_folds = {}
for r in RECORDS:
    pid = db.loc[int(r["rid"]), "patient_id"]
    patient_folds.setdefault(pid, set()).add(r["fold"])
overlaps = sum(1 for fs_ in patient_folds.values() if len(fs_) > 1)

print(f"Loaded {len(RECORDS)} records ({skipped} skipped: no diagnostic superclass)")
print(f"Train (folds 1-8): {len(train_idx)}   Val (fold 9): {len(val_idx)}   Test (fold 10): {len(test_idx)}")
print(f"Patient-fold leak check: {overlaps} patients appear in more than one fold "
      f"({'OK — no leakage' if overlaps == 0 else 'WARNING'})")
print("\nMulti-label class counts (presence, records may carry several):")
for i, s in enumerate(SUPERCLASSES):
    n_tr = int(y_all[train_idx][:, i].sum())
    n_all = int(y_all[:, i].sum())
    print(f"  {s:5s}: train {n_tr:5d}   all {n_all:5d}   ({n_all / len(RECORDS) * 100:.1f}% presence)")
multi = (y_all.sum(axis=1) > 1).mean()
print(f"\nRecords carrying >1 superclass: {multi * 100:.1f}%")

POS_WEIGHT = (len(train_idx) - y_all[train_idx].sum(0)) / np.maximum(y_all[train_idx].sum(0), 1)
print("pos_weight (neg/pos per class):", dict(zip(SUPERCLASSES, np.round(POS_WEIGHT, 2))))

## 4. Signal cache (memmap) & DataLoaders

WFDB loading is the training bottleneck, so all `(12, 5000)` tensors are precomputed **once** into a float32 memmap (~5 GB on disk, paged into RAM lazily). Every epoch after the first then runs at GPU speed. The one-time pass takes ~6–10 min.

The cache is **validated before reuse** (record ids and shape must match this exact run) and **deleted if the build is interrupted** — a stale, half-written, or misaligned cache is the classic cause of validation AUROC stuck at exactly 0.5. The train loader applies light on-the-fly augmentation (±100 ms shift, gain jitter, noise); val/test loaders never augment.

In [ ]:
import wfdb

SIG_PATH = WORK / "ecg_signals_f32.npy"
META_PATH = WORK / "ecg_signals_meta.json"
N = len(RECORDS)
CACHE_RIDS = [r["rid"] for r in RECORDS]


def _cache_is_valid():
    # Reuse the cache ONLY when it was built for this exact record list (count
    # AND order). A cache from a different dataset/run misaligns signals with
    # labels, which silently trains to AUROC 0.5.
    if not (SIG_PATH.exists() and META_PATH.exists()):
        return False
    try:
        if json.loads(META_PATH.read_text()).get("rids") != CACHE_RIDS:
            print("Signal cache does not match the current records - rebuilding.")
            return False
        m = np.load(SIG_PATH, mmap_mode="r")
        if m.shape != (N, 12, 5000):
            print("Signal cache has the wrong shape - rebuilding.")
            return False
        return True
    except Exception:
        return False


if _cache_is_valid():
    sig_arr = np.load(SIG_PATH, mmap_mode="r")
    print(f"Reusing signal cache for {N} records.")
else:
    t0 = time.time()
    arr = None
    try:
        arr = np.lib.format.open_memmap(SIG_PATH, mode="w+", dtype=np.float32, shape=(N, 12, 5000))
        for i, rec in enumerate(RECORDS):
            # wfdb.rdsamp returns (signals, fields_dict) - the sample rate is
            # fields["fs"], NOT the second tuple element itself.
            raw, meta = wfdb.rdsamp(str(DATA_DIR / rec["filename_hr"]))
            seg = preprocess_ecg(raw, meta["fs"])[0]   # (5000, 12)
            arr[i] = seg.T.astype(np.float32)          # (12, 5000)
            if (i + 1) % 2000 == 0:
                arr.flush()
                print(f"  {i + 1}/{N} records  ({time.time() - t0:.0f}s elapsed)")
        arr.flush()
        del arr
        META_PATH.write_text(json.dumps({"rids": CACHE_RIDS}))
        sig_arr = np.load(SIG_PATH, mmap_mode="r")
        print(f"Cached {N} signals in {time.time() - t0:.0f}s -> {SIG_PATH}")
    except Exception:
        # open_memmap(mode="w+") creates the FULL file up front, so an
        # interrupted build leaves zeros on disk. A later run would reuse that
        # poison cache and train to AUROC 0.5 - always remove it on failure.
        if arr is not None:
            del arr
        for p in (SIG_PATH, META_PATH):
            try:
                p.unlink()
            except OSError:   # incl. FileNotFoundError; never mask the real error
                pass
        raise

# Sanity: flat / non-finite cached signals also pin AUROC at 0.5 - catch it here.
_rng = np.random.default_rng(0)
for k in _rng.choice(N, size=min(25, N), replace=False):
    s = np.asarray(sig_arr[int(k)])
    assert np.isfinite(s).all() and s.std() > 0.05, (
        f"Cached signal {int(k)} looks flat or invalid - delete {SIG_PATH} and re-run this cell.")


class PTBXLMultilabel(Dataset):
    def __init__(self, sig_arr, y_all, idxs, augment=False):
        self.sig, self.y, self.idxs = sig_arr, y_all, idxs
        self.augment = augment   # train-time only; never for val/test

    def __len__(self):
        return len(self.idxs)

    def __getitem__(self, i):
        j = self.idxs[i]
        sig = np.array(self.sig[j], dtype=np.float32)   # writable copy from the memmap
        if self.augment:
            # Light, morphology-preserving augmentation:
            shift = np.random.randint(-50, 51)          # +-100 ms circular time shift
            sig = np.roll(sig, shift, axis=1)
            sig *= np.random.uniform(0.9, 1.1)          # random gain
            sig = sig + (0.05 * np.random.randn(*sig.shape)).astype(np.float32)  # noise
        return (
            torch.from_numpy(np.ascontiguousarray(sig)).float(),
            torch.from_numpy(self.y[j]).float(),
        )


BATCH_SIZE = 64
train_loader = DataLoader(PTBXLMultilabel(sig_arr, y_all, train_idx, augment=True), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(PTBXLMultilabel(sig_arr, y_all, val_idx),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(PTBXLMultilabel(sig_arr, y_all, test_idx),  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

xb, yb = next(iter(test_loader))
print(f"Batch shapes - signals: {tuple(xb.shape)}  labels: {tuple(yb.shape)}  (multi-hot)")


## 5. Stage 1b — 1D SE-ResNet architecture

Residual blocks with **squeeze-and-excitation channel attention** (cheap, reliably worth 1–2 AUROC points on 12-lead ECG) → global average pooling → dense → **5 sigmoid outputs** (one probability per superclass — multi-label, not softmax). Output is raw logits; `BCEWithLogitsLoss` applies the sigmoid internally.

In [ ]:
class SEBlock(nn.Module):
    # Squeeze-and-excitation channel attention: global-pool the time axis,
    # learn per-channel weights, rescale. Cheap and effective on ECG leads.
    def __init__(self, c, r=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(c, max(c // r, 4)),
            nn.ReLU(inplace=True),
            nn.Linear(max(c // r, 4), c),
            nn.Sigmoid(),
        )

    def forward(self, x):                     # x: (B, C, T)
        w = x.mean(dim=2)                     # (B, C)
        return x * self.fc(w).unsqueeze(-1)


class ResBlock(nn.Module):
    # conv-bn-relu-conv-bn + SE attention + identity skip.
    def __init__(self, cin, cout, kernel=15, stride=1):
        super().__init__()
        pad = kernel // 2
        self.conv1 = nn.Conv1d(cin, cout, kernel, stride=stride, padding=pad, bias=False)
        self.bn1 = nn.BatchNorm1d(cout)
        self.conv2 = nn.Conv1d(cout, cout, kernel, stride=1, padding=pad, bias=False)
        self.bn2 = nn.BatchNorm1d(cout)
        self.se = SEBlock(cout)
        self.act = nn.ReLU(inplace=True)
        self.downsample = None
        if stride != 1 or cin != cout:
            self.downsample = nn.Sequential(
                nn.Conv1d(cin, cout, 1, stride=stride, bias=False),
                nn.BatchNorm1d(cout),
            )

    def forward(self, x):
        identity = x if self.downsample is None else self.downsample(x)
        out = self.act(self.bn1(self.conv1(x)))
        out = self.se(self.bn2(self.conv2(out)))
        return self.act(out + identity)


class ResNet1D(nn.Module):
    # 1D SE-ResNet for multi-label ECG classification.
    # Input:  (batch, 12, 5000)   Output: (batch, 5) logits
    def __init__(self, num_leads=12, num_classes=5, base=64):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(num_leads, base, 15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(base),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),
        )
        self.stage1 = nn.Sequential(ResBlock(base, base),       ResBlock(base, base))
        self.stage2 = nn.Sequential(ResBlock(base, base * 2, stride=2),   ResBlock(base * 2, base * 2))
        self.stage3 = nn.Sequential(ResBlock(base * 2, base * 3, stride=2), ResBlock(base * 3, base * 3))
        self.stage4 = nn.Sequential(ResBlock(base * 3, base * 4, stride=2), ResBlock(base * 4, base * 4))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(base * 4, num_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        return self.head(x)


_test = ResNet1D()
_out = _test(torch.randn(2, 12, 5000))
print(f"Model OK - output shape: {tuple(_out.shape)}  |  Parameters: {sum(p.numel() for p in _test.parameters()):,}")
del _test, _out

## 6. Stage 1c — Training (BCEWithLogitsLoss, official folds)

Multi-label objective with per-class `pos_weight`, AdamW with **linear warmup + cosine decay**, mixed precision, gradient clipping, and **train-time augmentation**. Early stopping on **validation macro-AUROC** (fold 9) with patience 10 — AUROC often plateaus for several epochs before improving again, so 6 was too tight. The best checkpoint is restored at the end.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

EPOCHS = 40
WARMUP_EPOCHS = 2
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 10          # AUROC can plateau for several epochs before improving
GRAD_CLIP = 1.0
USE_AMP = DEVICE.type == "cuda"

# A class with no positives in train or val pins its AUROC at 0.5 - fail loudly.
for i, s in enumerate(SUPERCLASSES):
    n_tr = int(y_all[train_idx][:, i].sum())
    n_va = int(y_all[val_idx][:, i].sum())
    assert n_tr > 0 and n_va > 0, (
        f"{s}: {n_tr} train / {n_va} val positives - labels are broken, fix the data first")

model = ResNet1D(num_leads=12, num_classes=len(SUPERCLASSES)).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(POS_WEIGHT, dtype=torch.float32, device=DEVICE)
)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


def lr_lambda(epoch):
    # Linear warmup (stabilises the first epochs) then cosine decay to ~0.
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    prog = (epoch - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1)
    return 0.5 * (1.0 + math.cos(math.pi * prog))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler(device=DEVICE.type, enabled=USE_AMP)


@torch.no_grad()
def collect_probs(loader):
    # Sigmoid probabilities + labels for a whole loader, in loader order.
    model.eval()
    probs, labels = [], []
    for xb, yb in loader:
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            logits = model(xb.to(DEVICE, non_blocking=True))
        probs.append(torch.sigmoid(logits.float()).cpu())
        labels.append(yb.cpu())
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()


history = {"train_loss": [], "val_macro_auroc": [], "lr": []}
best_auroc, best_state, wait = -1.0, None, 0

for epoch in range(EPOCHS):
    model.train()
    t0, total, nb = time.time(), 0.0, 0
    for xb, yb in train_loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            loss = criterion(model(xb), yb)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        total += loss.item()
        nb += 1

    scheduler.step()

    val_p, val_y = collect_probs(val_loader)

    # AUROC is only defined where the val fold has both classes; macro over the rest.
    auroc_map = {}
    for i, s in enumerate(SUPERCLASSES):
        if len(np.unique(val_y[:, i])) > 1:
            auroc_map[s] = roc_auc_score(val_y[:, i], val_p[:, i])
    macro = float(np.mean(list(auroc_map.values()))) if auroc_map else 0.5

    history["train_loss"].append(total / max(nb, 1))
    history["val_macro_auroc"].append(macro)
    history["lr"].append(optimizer.param_groups[0]["lr"])

    saved = ""
    if macro > best_auroc + 1e-4:
        best_auroc, saved = macro, " *"
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1

    per_class = " ".join(f"{s}={auroc_map.get(s, float('nan')):.3f}" for s in SUPERCLASSES)
    print(f"Epoch {epoch + 1:3d}/{EPOCHS} | loss {total / max(nb, 1):.4f} | "
          f"val macro-AUROC {macro:.4f}{saved} | {per_class} | "
          f"lr {optimizer.param_groups[0]['lr']:.2e} | {time.time() - t0:.0f}s")

    if wait >= PATIENCE:
        print(f"Early stopping (no improvement for {PATIENCE} epochs).")
        break

if best_state is not None:
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), WORK / "resnet1d_multilabel_best.pth")
    print(f"\nRestored best checkpoint (val macro-AUROC {best_auroc:.4f}) -> {WORK / 'resnet1d_multilabel_best.pth'}")
else:
    print("\nNo best state saved (model never improved).")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"])
axes[0].set_title("Training loss (BCEWithLogits)")
axes[0].set_xlabel("Epoch")
axes[1].plot(history["val_macro_auroc"], color="tab:red")
axes[1].set_title("Validation macro-AUROC (fold 9)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylim(0.5, 1.0)
plt.tight_layout()
plt.savefig(WORK / "training_curves.png", dpi=130)
plt.show()

## 7. Stage 1d — Threshold tuning & test-fold evaluation

Per-class decision thresholds are tuned on the **validation fold** (maximize per-class F1) — never on test. Test-fold metrics: **AUROC, AUPRC, F1, sensitivity, specificity** per class + macro averages.

In [ ]:
from sklearn.metrics import average_precision_score, f1_score

val_probs, val_y = collect_probs(val_loader)
THRESHOLDS = {}
for i, s in enumerate(SUPERCLASSES):
    best_f1, best_t = -1.0, 0.5
    for t in np.arange(0.10, 0.91, 0.05):
        f1 = f1_score(val_y[:, i], val_probs[:, i] >= t, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    THRESHOLDS[s] = best_t
print("Val-tuned thresholds:", THRESHOLDS)

test_probs, test_y = collect_probs(test_loader)
test_records = [r for r in RECORDS if r["fold"] == TEST_FOLD]
PROBS_BY_RID = {r["rid"]: test_probs[k] for k, r in enumerate(test_records)}

rows = []
for i, s in enumerate(SUPERCLASSES):
    p, y = test_probs[:, i], test_y[:, i]
    pred = p >= THRESHOLDS[s]
    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    rows.append({
        "class": s,
        "AUROC": roc_auc_score(y, p),
        "AUPRC": average_precision_score(y, p),
        "F1": f1_score(y, pred, zero_division=0),
        "sens": tp / max(tp + fn, 1),
        "spec": tn / max(tn + fp, 1),
        "n_pos": int(y.sum()),
    })
metrics_df = pd.DataFrame(rows).set_index("class").round(4)
macro = metrics_df.mean(numeric_only=True).to_dict()
metrics_df.loc["macro"] = {k: round(v, 4) for k, v in macro.items() if k != "n_pos"}
print("\nTest fold (fold 10) — per-class metrics @ val-tuned thresholds:")
display(metrics_df)

with open(WORK / "thresholds.json", "w") as f:
    json.dump(THRESHOLDS, f, indent=2)
print(f"\nSaved thresholds → {WORK / 'thresholds.json'}")

## 8. Stage 2 — Signal measurement extraction

Every number below is **computed from the waveform or read from PTB-XL's validated annotations** — the LLM never gets to guess any of them. Anything that cannot be measured is reported as `null` (unavailable); nothing is ever filled in.

| Measurement | Method |
|---|---|
| Heart rate | R-peak detection on lead II (QRS-energy envelope + adaptive threshold) → median RR |
| Rhythm | PTB-XL `scp_codes` rhythm entries (validated dataset annotations) |
| PR / QRS / QT / QTc | Onset/offset detection on the **median beat template**; Bazett QT correction |
| P / QRS / T axis | Net deflection area of the median beat in leads I & aVF (Einthoven geometry); QRS axis cross-checked against PTB-XL's `heart_axis` annotation |

> Intervals are timing-based, so they are valid on the z-scored cache too; **axes need true mV amplitudes** (per-lead z-scoring would distort them), so the sample cases are re-loaded in mV.

In [ ]:
import math
from scipy.signal import find_peaks

LEAD_II, LEAD_I, LEAD_AVF = 1, 0, 5   # PTB-XL lead order: I II III aVR aVL aVF V1..V6

# Rhythm codes -> readable labels (dataset annotations, not model output)
RHYTHM_READABLE = {
    "SINUS": "sinus rhythm",
    "AFIB": "atrial fibrillation",
    "AFLT": "atrial flutter",
    "SVTAC": "supraventricular tachycardia",
    "PSVT": "paroxysmal supraventricular tachycardia",
    "AT": "atrial tachycardia",
    "SARRH": "sinus arrhythmia",
    "SBRAD": "sinus bradycardia",
    "STACH": "sinus tachycardia",
    "PACE": "paced rhythm",
}


def detect_r_peaks(sig, fs=500):
    # R-peak candidates on one lead: QRS-energy envelope + adaptive threshold.
    deriv = np.abs(np.gradient(sig))
    win = max(1, int(0.08 * fs))
    env = np.convolve(deriv, np.ones(win) / win, mode="same")
    if env.std() < 1e-9:
        return np.array([], dtype=int)
    peaks, _ = find_peaks(env, height=env.mean() + 2.0 * env.std(), distance=int(0.25 * fs))
    half, out = int(0.08 * fs), []
    for p in peaks:
        lo, hi = max(0, p - half), min(len(sig), p + half)
        out.append(lo + int(np.argmax(sig[lo:hi])))
    return np.array(sorted(set(out)), dtype=int)


def median_beat(sig, r_peaks, fs=500):
    # Median-aligned beat template on one lead. Returns (template, r_index) or (None, None).
    pre, post = int(0.30 * fs), int(0.45 * fs)
    beats = [sig[p - pre: p + post] for p in r_peaks if p - pre >= 0 and p + post <= len(sig)]
    if len(beats) < 3:
        return None, None
    return np.median(np.stack(beats), axis=0), pre


def _q_onset(tmpl, r_idx, baseline, noise, fs):
    # Walk backwards from R; the onset is where the signal enters the noise band
    # for a SUSTAINED run (>=30 ms) - brief cancellation dips (e.g. the Q wave
    # cancelling the R tail) are skipped over.
    sustain = int(0.03 * fs)
    i = r_idx - 2
    stop = max(1, r_idx - int(0.12 * fs))
    while i > stop:
        if abs(tmpl[i] - baseline) <= noise:
            j = max(0, i - sustain)
            if np.all(np.abs(tmpl[j:i] - baseline) <= noise):
                return i
        i -= 1
    return None


def _qrs_offset(tmpl, r_idx, baseline, noise, fs):
    # Walk forward from R; the offset is where the signal enters the noise band
    # for a SUSTAINED run. None if it never settles within 140 ms (e.g. marked
    # ST deviation) - QRS then stays unavailable rather than wrong.
    sustain = int(0.03 * fs)
    i = r_idx + 2
    stop = min(len(tmpl) - 1, r_idx + int(0.14 * fs))
    while i < stop:
        if abs(tmpl[i] - baseline) <= noise:
            seg = np.abs(tmpl[i: i + sustain] - baseline)
            if len(seg) == sustain and np.all(seg <= noise):
                return i
        i += 1
    return None


def _t_offset(tmpl, q_off, baseline, noise, fs):
    # Offset of the T wave: first sustained (>=60 ms) return to baseline AFTER the T peak.
    lo = q_off + int(0.05 * fs)
    hi = min(len(tmpl) - 1, q_off + int(0.48 * fs))
    if hi <= lo:
        return None
    dev = np.abs(tmpl[lo:hi] - baseline)
    t_peak = lo + int(np.argmax(dev))
    if dev[t_peak - lo] <= noise:
        return None  # no discernible T wave -> QT unavailable
    sustain = int(0.06 * fs)
    i = t_peak
    while i < hi:
        if abs(tmpl[i] - baseline) <= noise:
            seg = np.abs(tmpl[i: i + sustain] - baseline)
            if len(seg) == sustain and np.all(seg <= noise):
                return i
        i += 1
    return None  # T wave never settles back within the window


def _p_onset(tmpl, q_on, baseline, noise, fs):
    # Onset of the P wave before QRS; None when no discernible P (e.g. atrial fibrillation).
    # P waves are much smaller than R, so they get their own gentler noise band.
    p_noise = noise * 0.5
    hi = q_on - int(0.02 * fs)
    lo = max(1, q_on - int(0.22 * fs))
    if hi <= lo:
        return None
    dev = np.abs(tmpl[lo:hi] - baseline)
    p_peak = lo + int(np.argmax(dev))
    if dev[p_peak - lo] <= p_noise:
        return None
    sustain = int(0.04 * fs)
    i = p_peak
    while i > lo:
        if abs(tmpl[i] - baseline) <= p_noise:
            j = max(0, i - sustain)
            if np.all(np.abs(tmpl[j:i] - baseline) <= p_noise):
                return i
        i -= 1
    return None


def _axis_category(net_i, net_avf):
    # Hexaxial QRS axis from net deflection areas in leads I and aVF.
    if abs(net_i) < 1e-9 and abs(net_avf) < 1e-9:
        return None
    deg = math.degrees(math.atan2(net_avf, net_i))
    if -30 <= deg <= 90:
        return f"normal axis ({deg:.0f} deg)"
    if deg > 90:
        return f"right axis deviation ({deg:.0f} deg)"
    if deg < -90:
        return f"extreme axis ({deg:.0f} deg)"
    return f"left axis deviation ({deg:.0f} deg)"


def rhythm_from_metadata(rec):
    # Rhythm comes from PTB-XL's validated rhythm annotations, never from the LLM.
    for code in rec["scp_codes"]:
        if code in RHYTHM_READABLE:
            return RHYTHM_READABLE[code]
    return None


def readable_axis_annotation(val):
    if not isinstance(val, str) or not val.strip():
        return None
    v = val.strip().upper()
    if v in ("MID", "AXIS_NORMAL", "NORMAL"):
        return "normal axis (dataset annotation)"
    if v == "AXR" or "RIGHT" in v:
        return "right axis deviation (dataset annotation)"
    if v == "AXL" or "LEFT" in v:
        return "left axis deviation (dataset annotation)"
    if v == "AXIND" or "INDETERMINATE" in v:
        return "indeterminate axis (dataset annotation)"
    return f"{val.strip()} (dataset annotation)"


def measure_record(rec, sig, fs=500, in_mv=True):
    # Stage 2 for one record. `sig` is (samples, 12). Pass the raw mV signal
    # (in_mv=True) when axes matter: per-lead z-scoring distorts net areas.
    # Unmeasurable quantities stay None - they are never filled in.
    out = {"hr": None, "rhythm": rhythm_from_metadata(rec), "pr": None, "qrs": None,
           "qt": None, "qtc": None, "qrs_axis": None, "p_axis": None, "t_axis": None}
    try:
        r_peaks = detect_r_peaks(sig[:, LEAD_II], fs)
        if len(r_peaks) < 3:
            return out
        rr = np.diff(r_peaks) / fs
        rr = rr[(rr > 0.33) & (rr < 2.5)]
        if len(rr) == 0:
            return out
        out["hr"] = int(round(60.0 / float(np.median(rr))))

        tmpl, r_idx = median_beat(sig[:, LEAD_II], r_peaks, fs)
        if tmpl is None:
            return out
        baseline = float(np.median(tmpl[: int(0.06 * fs)]))
        noise = 0.15 * float(np.max(np.abs(tmpl - baseline)))
        # The median template suppresses beat-to-beat noise, so the QRS onset/offset
        # walks can use a much gentler band than the wave-presence checks - this is
        # what keeps small Q waves and S tails inside the measured QRS duration.
        qrs_noise = noise / 3.0

        q_on = _q_onset(tmpl, r_idx, baseline, qrs_noise, fs)
        q_off = _qrs_offset(tmpl, r_idx, baseline, qrs_noise, fs)
        p_on = t_off = None
        if q_on is not None and q_off is not None:
            out["qrs"] = int(round((q_off - q_on) / fs * 1000))
            p_on = _p_onset(tmpl, q_on, baseline, noise, fs)
            if p_on is not None:
                out["pr"] = int(round((q_on - p_on) / fs * 1000))
            t_off = _t_offset(tmpl, q_off, baseline, noise, fs)
            if t_off is not None:
                qt_ms = (t_off - q_on) / fs * 1000
                out["qt"] = int(round(qt_ms))
                out["qtc"] = int(round(qt_ms / math.sqrt(float(np.median(rr)))))

        if in_mv and q_on is not None and q_off is not None:
            tmpl_i, _ = median_beat(sig[:, LEAD_I], r_peaks, fs)
            tmpl_f, _ = median_beat(sig[:, LEAD_AVF], r_peaks, fs)
            if tmpl_i is not None and tmpl_f is not None:
                net_i = float(np.sum(tmpl_i[q_on:q_off]))
                net_f = float(np.sum(tmpl_f[q_on:q_off]))
                # Prefer PTB-XL's validated annotation; fall back to the computed axis.
                out["qrs_axis"] = readable_axis_annotation(rec["heart_axis"]) or _axis_category(net_i, net_f)
                if p_on is not None:
                    plo, phi = max(0, q_on - int(0.18 * fs)), q_on - int(0.03 * fs)
                    out["p_axis"] = _axis_category(float(np.sum(tmpl_i[plo:phi])),
                                                   float(np.sum(tmpl_f[plo:phi])))
                if t_off is not None:
                    tlo, thi = q_off + int(0.05 * fs), t_off
                    out["t_axis"] = _axis_category(float(np.sum(tmpl_i[tlo:thi])),
                                                   float(np.sum(tmpl_f[tlo:thi])))
    except Exception:
        pass  # measurement failure -> stays None (unavailable, never invented)
    return out


print("Stage 2 measurement functions defined.")

In [ ]:
import time

# Apply Stage 2 to the whole test fold. Timing-based intervals are valid on the
# z-scored cache; axes are skipped there (in_mv=False) and computed for the
# narrative sample in Cell 22 from the true mV signal.
t0 = time.time()
MEAS_BY_RID = {}
for rec in test_records:
    MEAS_BY_RID[rec["rid"]] = measure_record(rec, sig_arr[REC_INDEX[rec["rid"]]].T, in_mv=False)
print(f"Measured {len(MEAS_BY_RID)} test-fold records in {time.time() - t0:.1f}s")

avail = {k: 0 for k in ("hr", "pr", "qrs", "qt", "qtc")}
for m in MEAS_BY_RID.values():
    for k in avail:
        avail[k] += m[k] is not None
print("Measurement availability (test fold) - failures stay unavailable by design:")
for k, v in avail.items():
    print(f"  {k:4s}: {v:5d}/{len(MEAS_BY_RID)}  ({v / len(MEAS_BY_RID) * 100:.1f}%)")

# Agreement preview vs PTB-XL's validated hr annotation (full stats in Cell 28)
pairs = []
for rec in test_records:
    m = MEAS_BY_RID[rec["rid"]]
    if m["hr"] is not None and rec["hr"] is not None:
        pairs.append((m["hr"], rec["hr"]))
if pairs:
    errs = [abs(c - g) for c, g in pairs]
    print(f"\nComputed HR vs PTB-XL annotation (n={len(pairs)}): "
          f"MAE {np.mean(errs):.1f} bpm | {np.mean([e <= 5 for e in errs]) * 100:.1f}% within 5 bpm")

## 9. Stage 3 — Structured case object

One JSON per record merging the three honest sources — **model predictions** (Stage 1), **signal measurements** (Stage 2), and **PTB-XL metadata** (age, sex, readable diagnoses). This object is the *only* thing the LLM will ever see.

- `findings` are auto-derived observations (rate/QRS/QTc statements + thresholded model outputs, clearly attributed).
- `diagnosis` lists the dataset's own annotations (`scp_codes` → readable names) — ground truth for evaluation, and clearly labelled as annotations in the report.

In [ ]:
# SCP code -> readable diagnosis (each key defined exactly ONCE - see project
# post-mortem on silent dict-key overwrites; unknown codes pass through as-is)
SCP_READABLE = {
    # NORM
    "NORM": "Normal ECG",
    # MI
    "AMI": "Anterior myocardial infarction",
    "ASMI": "Anteroseptal myocardial infarction",
    "IMI": "Inferior myocardial infarction",
    "LMI": "Lateral myocardial infarction",
    "PMI": "Posterior myocardial infarction",
    # CD
    "1AVB": "First-degree AV block",
    "2AVB": "Second-degree AV block",
    "3AVB": "Third-degree (complete) AV block",
    "CRBBB": "Complete right bundle branch block",
    "CLBBB": "Complete left bundle branch block",
    "IRBBB": "Incomplete right bundle branch block",
    "ILBBB": "Incomplete left bundle branch block",
    "LAFB": "Left anterior fascicular block",
    "LPFB": "Left posterior fascicular block",
    "IVCD": "Non-specific intraventricular conduction delay",
    # HYP
    "LVH": "Left ventricular hypertrophy",
    "RVH": "Right ventricular hypertrophy",
    "LAH": "Left atrial enlargement",
    "LAE": "Left atrial enlargement",
    "RAH": "Right atrial enlargement",
    "RAE": "Right atrial enlargement",
    # STTC
    "NST_": "Non-specific ST-T abnormality",
    "DIG": "Digitalis effect on ST-T",
    "EL": "Electrolyte-related ST-T change",
}


def readable_scp(code):
    if code in SCP_READABLE:
        return SCP_READABLE[code]
    if code.startswith("ISC"):
        return f"Ischemic ST-segment change ({code})"
    return code


def derive_findings(ecg, probs):
    # Observation-level findings, restating only values already in the case object.
    f = []
    hr = ecg.get("heart_rate")
    if hr is None:
        f.append("Heart rate not measurable from this recording")
    elif hr < 60:
        f.append(f"Bradycardia at {hr} bpm")
    elif hr > 100:
        f.append(f"Tachycardia at {hr} bpm")
    else:
        f.append(f"Heart rate {hr} bpm, within normal limits")
    qrs = ecg.get("qrs")
    if qrs is not None:
        if qrs >= 120:
            f.append(f"Wide QRS complex ({qrs} ms)")
        elif qrs <= 70:
            f.append(f"Narrow QRS complex ({qrs} ms)")
    qtc = ecg.get("qtc")
    if qtc is not None:  # simplified thresholds for demonstration
        if qtc >= 470:
            f.append(f"Prolonged QTc ({qtc} ms)")
        elif qtc <= 350:
            f.append(f"Short QTc ({qtc} ms)")
    pr = ecg.get("pr")
    if pr is not None and pr >= 200:
        f.append(f"Prolonged PR interval ({pr} ms)")
    ax = ecg.get("qrs_axis")
    if isinstance(ax, str) and "deviation" in ax:
        f.append(ax[0].upper() + ax[1:])
    flagged = [(s, p) for s, p in zip(SUPERCLASSES, probs) if p >= THRESHOLDS[s]]
    for s, p in flagged:
        f.append(f"Model flags {s} (probability {p:.2f}, threshold {THRESHOLDS[s]:.2f})")
    if not flagged:
        f.append("No superclass exceeds its decision threshold")
    return f


def build_case(rec, probs, meas):
    # Stage 3: the structured case object - the LLM's ONLY input.
    ecg = {
        "heart_rate": meas["hr"],
        "rhythm": meas["rhythm"],
        "pr": meas["pr"], "qrs": meas["qrs"], "qt": meas["qt"], "qtc": meas["qtc"],
        "qrs_axis": meas["qrs_axis"], "p_axis": meas["p_axis"], "t_axis": meas["t_axis"],
    }
    diagnosis = [readable_scp(c) for c, _ in sorted(rec["scp_codes"].items(), key=lambda kv: -kv[1])
                 if c in CODE_TO_SUPERCLASS]
    return {
        "case_id": rec["rid"],
        "patient": {"age": rec["age"], "sex": rec["sex"]},
        "ecg": ecg,
        "findings": derive_findings(ecg, probs),
        "diagnosis": diagnosis,
        "model_predictions": {s: round(float(p), 4) for s, p in zip(SUPERCLASSES, probs)},
    }


def pick_narrative_sample():
    # Deterministic, diverse sample: normal, single MI, multi-label MI, CD, HYP.
    profiles = [
        ("NORM only", lambda r: r["supers"] == ["NORM"]),
        ("MI only", lambda r: r["supers"] == ["MI"]),
        ("MI multi-label", lambda r: "MI" in r["supers"] and len(r["supers"]) > 1),
        ("CD", lambda r: "CD" in r["supers"]),
        ("HYP", lambda r: "HYP" in r["supers"]),
    ]
    chosen = []
    for name, match in profiles:
        for rec in test_records:
            if rec not in chosen and match(rec):
                print(f"  sample [{name:14s}] -> record {rec['rid']}  GT={rec['supers']}")
                chosen.append(rec)
                break
    return chosen


print("Selecting narrative sample from the test fold:")
NARR_SAMPLE = pick_narrative_sample()

# Build the sample cases from the true mV signal so the axes are valid.
CASES = {}
for rec in NARR_SAMPLE:
    raw, meta = wfdb.rdsamp(str(DATA_DIR / rec["filename_hr"]))
    raw_mv = bandpass_filter(raw, meta["fs"])   # filtered but NOT per-lead normalized
    CASES[rec["rid"]] = build_case(rec, PROBS_BY_RID[rec["rid"]],
                                   measure_record(rec, raw_mv, fs=int(meta["fs"]), in_mv=True))

with open(WORK / "structured_cases.json", "w") as f:
    json.dump(list(CASES.values()), f, indent=2)
print(f"\nBuilt {len(CASES)} structured case objects -> {WORK / 'structured_cases.json'}")
print("\nExample case JSON (this is ALL the LLM will ever see):")
print(json.dumps(list(CASES.values())[0], indent=2))

## 10. Stage 4 — LLM narrative layer (phrasing only, enforced)

The LLM's **only input is the Stage 3 JSON — never the waveform**. Its job is to *phrase* the structured data into the five report sections: Clinical Presentation, ECG Findings, Diagnosis, Interpretation, Key Teaching Points.

Two guardrails:

1. **Prompt-level**: strict rules — only facts from the JSON, `null` → "unavailable", no invented or rounded numbers.
2. **Post-hoc numeric validation**: every number extracted from the generated text is checked against the set of numbers present in the input JSON (with formatting variants: `0.87`/`0.8712`/`87%`). A draft with violations is regenerated once, then discarded in favour of a deterministic template phraser.

With a `GEMINI_API_KEY` in Kaggle Secrets the real LLM is used; without a key the template phraser keeps the pipeline runnable end-to-end.

In [ ]:
import requests

LLM_MODEL = "gemini-2.0-flash"   # change if your quota/API requires another model

try:
    from kaggle_secrets import UserSecretsClient
    GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
HAS_LLM = bool(GEMINI_API_KEY)
print("Narrative layer:", f"Gemini ({LLM_MODEL})" if HAS_LLM
      else "no API key - deterministic template phraser (pipeline still runs)")

SYSTEM_NARRATIVE = (
    "You are a medical writer producing the narrative sections of a clinical ECG report. "
    "You will receive ONE structured JSON case object and must rephrase it into prose. "
    "STRICT RULES: "
    "(1) Use ONLY the facts, numbers and diagnoses present in the JSON. Never invent, estimate "
    "or round any value. "
    "(2) Where a measurement is null, state that it is unavailable - do not guess. "
    "(3) No numbered lists and no headings other than the five required section headers. "
    "(4) Model probabilities must be described as statistical model outputs, not as diagnoses. "
    "(5) Keep the total under 350 words. "
    "Output EXACTLY these five sections, each starting with its header on its own line: "
    "Clinical Presentation | ECG Findings | Diagnosis | Interpretation | Key Teaching Points."
)


def call_gemini(case_json_str):
    # One generateContent call; temperature kept low for faithful phrasing.
    resp = requests.post(
        "https://generativelanguage.googleapis.com/v1beta/models/" + LLM_MODEL + ":generateContent",
        headers={"x-goog-api-key": GEMINI_API_KEY, "Content-Type": "application/json"},
        json={
            "contents": [{"role": "user", "parts": [{"text": SYSTEM_NARRATIVE + "\n\nCASE JSON:\n" + case_json_str}]}],
            "generationConfig": {"temperature": 0.2, "maxOutputTokens": 1024},
        },
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()["candidates"][0]["content"]["parts"][0]["text"].strip()


NUM_RE = re.compile(r"\d+(?:\.\d+)?")


def _allowed_numbers(case):
    # Every number the narrative may legitimately contain: all numeric values in
    # the case JSON plus their formatting variants (rounded, percent-phrased),
    # and any numbers embedded in its string values (findings text, case id...).
    allowed = {"10", "12"}  # structural: "10-second", "12-lead" are not measurements

    def add(v):
        variants = [f"{v:g}", f"{v:.2f}", f"{v:.1f}", f"{v:.0f}"]
        if 0.0 < v < 1.0:  # probabilities may be phrased as percentages
            pv = v * 100
            variants += [f"{pv:g}", f"{pv:.2f}", f"{pv:.1f}", f"{pv:.0f}"]
        allowed.update(variants)
        if v == int(v):
            allowed.add(str(int(v)))

    def walk(o):
        if isinstance(o, bool) or o is None:
            return
        if isinstance(o, (int, float)):
            add(float(o))
        elif isinstance(o, str):
            for tok in NUM_RE.findall(o):
                allowed.add(tok)
        elif isinstance(o, dict):
            for x in o.values():
                walk(x)
        elif isinstance(o, (list, tuple)):
            for x in o:
                walk(x)

    walk(case)
    return allowed


def numeric_violations(case, text):
    # Numbers in the text that cannot be traced back to the case JSON.
    allowed = _allowed_numbers(case)
    bad = []
    for tok in NUM_RE.findall(text):
        if tok in allowed:
            continue
        try:
            v = float(tok)
        except ValueError:
            continue
        if f"{v:g}" not in allowed:
            bad.append(tok)
    return bad


def template_narrative(case):
    # Deterministic fallback: the same five sections, assembled without any LLM.
    ecg, pat = case["ecg"], case["patient"]

    def s(v, unit=""):
        return "unavailable" if v is None else f"{v}{unit}"

    age = s(pat.get("age"), " years")
    sex = pat.get("sex") or "unavailable"
    preds = ", ".join(f"{k} {v:.2f}" for k, v in case["model_predictions"].items())
    lines = [
        "Clinical Presentation",
        f"A {age}-old {sex} patient; 10-second 12-lead ECG recording (case {case['case_id']}, PTB-XL).",
        "",
        "ECG Findings",
        f"Rhythm: {ecg.get('rhythm') or 'not annotated'}. Heart rate {s(ecg.get('heart_rate'), ' bpm')}. "
        f"PR interval {s(ecg.get('pr'), ' ms')}; QRS duration {s(ecg.get('qrs'), ' ms')}; "
        f"QT {s(ecg.get('qt'), ' ms')} with QTc {s(ecg.get('qtc'), ' ms')}. "
        f"QRS axis: {ecg.get('qrs_axis') or 'unavailable'}.",
    ]
    lines += ["- " + f for f in case["findings"]]
    lines += ["", "Diagnosis"]
    if case["diagnosis"]:
        lines += ["- " + d + " (dataset annotation)" for d in case["diagnosis"]]
    else:
        lines.append("- No diagnostic annotations in the dataset for this record")
    lines += [
        "",
        "Interpretation",
        f"Multi-label classifier outputs: {preds}. Values above their tuned thresholds are reported "
        "as findings; these are statistical outputs, not diagnoses by themselves.",
        "",
        "Key Teaching Points",
        "- The report narrative is generated strictly from the structured case object; no values were added.",
        "- A multi-label ECG model scores five superclasses independently - combined findings are common.",
        "- Measurements reported as unavailable were not computable and were deliberately not filled in.",
    ]
    return "\n".join(lines)


def generate_narrative(case):
    # LLM phrasing with the numeric guardrail; deterministic fallback if unavailable/violating.
    case_str = json.dumps(case, indent=2)
    if not HAS_LLM:
        return template_narrative(case), []
    for attempt in range(2):
        try:
            prompt = case_str if attempt == 0 else case_str + (
                "\n\nIMPORTANT: your previous draft contained numbers that do not appear in the JSON "
                "above. Regenerate it using ONLY the exact values present in the JSON.")
            text = call_gemini(prompt)
        except Exception as e:
            print(f"  LLM call failed ({type(e).__name__}: {str(e)[:80]}) - template fallback")
            return template_narrative(case), []
        bad = numeric_violations(case, text)
        if not bad:
            return text, []
    print("  two drafts violated the numeric guardrail - template fallback")
    return template_narrative(case), bad


NARRATIVES, GUARD = {}, {}
for rid, case in CASES.items():
    NARRATIVES[rid], GUARD[rid] = generate_narrative(case)
    src = "LLM" if HAS_LLM else "template"
    print(f"{rid}: {len(NARRATIVES[rid].split())} words ({src})")

with open(WORK / "narratives.json", "w") as f:
    json.dump(NARRATIVES, f, indent=2)
print(f"\nSaved narratives -> {WORK / 'narratives.json'}")

## 11. Stage 5 — One-page PDF report per case

Each PDF contains: patient info, the ECG measurements table, findings, the (annotated) diagnosis, model prediction confidences with thresholds, the narrative from Stage 4, and **source attribution to PTB-XL / PhysioNet** (Wagner et al., *Scientific Data*, 2020). Generated with `fpdf2`.

In [ ]:
from fpdf import FPDF

SECTION_HEADERS = ("Clinical Presentation", "ECG Findings", "Diagnosis",
                   "Interpretation", "Key Teaching Points")


def _lat(s):
    # fpdf2 core fonts are latin-1; replace anything outside it.
    return str(s).encode("latin-1", "replace").decode("latin-1")


def build_pdf(case, narrative, out_path):
    pdf = FPDF()
    pdf.set_auto_page_break(True, margin=12)
    pdf.add_page()

    pdf.set_font("Helvetica", "B", 13)
    pdf.cell(0, 6, "Clinical ECG Report", new_x="LMARGIN", new_y="NEXT", align="C")
    pdf.set_font("Helvetica", "I", 7)
    pdf.cell(0, 4, _lat("Structured pipeline: signal processing + multi-label model + narrative layer "
                        "- source: PTB-XL / PhysioNet"), new_x="LMARGIN", new_y="NEXT", align="C")
    pdf.ln(2)

    # Patient & measurements table (two columns)
    pdf.set_font("Helvetica", "B", 9)
    pdf.cell(0, 5, "Patient & Measurements", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("Helvetica", "", 8)

    def s(v, unit=""):
        return "unavailable" if v is None else f"{v}{unit}"

    rows = [
        ("Case ID", case["case_id"]),
        ("Age", s(case["patient"]["age"], " y")),
        ("Sex", case["patient"]["sex"] or "unavailable"),
        ("Rhythm", case["ecg"]["rhythm"] or "not annotated"),
        ("Heart rate", s(case["ecg"]["heart_rate"], " bpm")),
        ("PR interval", s(case["ecg"]["pr"], " ms")),
        ("QRS duration", s(case["ecg"]["qrs"], " ms")),
        ("QT / QTc", f"{s(case['ecg']['qt'], ' ms')} / {s(case['ecg']['qtc'], ' ms')}"),
        ("QRS axis", case["ecg"]["qrs_axis"] or "unavailable"),
        ("P / T axis", f"{case['ecg']['p_axis'] or 'unavailable'} / {case['ecg']['t_axis'] or 'unavailable'}"),
    ]
    colw = (pdf.w - 2 * pdf.l_margin) / 2
    for i in range(0, len(rows), 2):
        for label, val in rows[i: i + 2]:
            pdf.cell(colw, 4.6, _lat(f"{label}: {val}"), border=1)
        pdf.ln()
    pdf.ln(1)

    pdf.set_font("Helvetica", "B", 9)
    pdf.cell(0, 5, "Findings", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("Helvetica", "", 8)
    for fnd in case["findings"]:
        pdf.multi_cell(0, 4, _lat("- " + fnd))
    pdf.ln(1)

    pdf.set_font("Helvetica", "B", 9)
    pdf.cell(0, 5, "Diagnosis (PTB-XL annotations)", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("Helvetica", "", 8)
    if case["diagnosis"]:
        for d in case["diagnosis"]:
            pdf.multi_cell(0, 4, _lat("- " + d))
    else:
        pdf.multi_cell(0, 4, "- No diagnostic annotations")
    pdf.ln(1)

    pdf.set_font("Helvetica", "B", 9)
    pdf.cell(0, 5, "Model Prediction Confidences (multi-label)", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("Helvetica", "", 8)
    for k, v in case["model_predictions"].items():
        pdf.cell(0, 4, _lat(f"{k:5s}: {v:.2f}   (threshold {THRESHOLDS[k]:.2f})"),
                 new_x="LMARGIN", new_y="NEXT")
    pdf.ln(1)

    pdf.set_font("Helvetica", "B", 9)
    pdf.cell(0, 5, "Interpretive Report", new_x="LMARGIN", new_y="NEXT")
    for line in narrative.splitlines():
        if not line.strip():
            pdf.ln(1)
            continue
        if line.strip() in SECTION_HEADERS:
            pdf.set_font("Helvetica", "B", 7.5)
            pdf.multi_cell(0, 3.6, _lat(line.strip()))
            pdf.set_font("Helvetica", "", 7.5)
        else:
            pdf.multi_cell(0, 3.6, _lat(line))

    pdf.set_font("Helvetica", "I", 6)
    pdf.multi_cell(0, 3, _lat("Source attribution: PTB-XL, PhysioNet (Wagner et al., Scientific Data, 2020). "
                              "Narrative text is machine-generated from the structured case object only; "
                              "numbers absent from that object are rejected by the validation layer. "
                              "Research use only - not for clinical decision-making."))
    pdf.output(str(out_path))


pdf_files = []
for rid, case in list(CASES.items())[:3]:
    p = WORK / f"case_{rid}.pdf"
    build_pdf(case, NARRATIVES[rid], p)
    pdf_files.append(p)
    print("wrote", p)

## 12. Evaluation

- **ML component** — done in Cell 17 (AUROC / AUPRC / F1 / sensitivity / specificity per class on the untouched test fold).
- **Metadata component** (below): HR accuracy against PTB-XL's validated `hr` annotation; superclass-set diagnosis accuracy (exact match + Jaccard); finding precision/recall of the model-derived findings against ground truth.
- **Narrative component**: guardrail violation counts + a clinician review rubric for manual expert scoring — deliberately *not* BLEU/ROUGE.

In [ ]:
# ── HR accuracy: computed HR vs PTB-XL's validated annotation ──
pairs = []
for rec in test_records:
    m = MEAS_BY_RID[rec["rid"]]
    if m["hr"] is not None and rec["hr"] is not None:
        pairs.append((m["hr"], rec["hr"]))
if pairs:
    errs = [abs(c - g) for c, g in pairs]
    print(f"HR accuracy vs annotation (n={len(pairs)}): MAE {np.mean(errs):.1f} bpm | "
          f"{np.mean([e <= 5 for e in errs]) * 100:.1f}% within 5 bpm | "
          f"{np.mean([e <= 10 for e in errs]) * 100:.1f}% within 10 bpm")
else:
    print("HR accuracy vs annotation: skipped - no annotated hr values available "
          "(older PTB-XL dataset version without the hr column)")

# ── Diagnosis accuracy: predicted superclass SET vs annotated superclass SET ──
exact, jacc = 0, []
for rec in test_records:
    probs = PROBS_BY_RID[rec["rid"]]
    pred = set(s for i, s in enumerate(SUPERCLASSES) if probs[i] >= THRESHOLDS[s])
    gold = set(rec["supers"])
    exact += pred == gold
    jacc.append(len(pred & gold) / len(pred | gold) if (pred | gold) else 1.0)
print(f"Superclass-set diagnosis: exact match {exact / len(test_records) * 100:.1f}% | "
      f"mean Jaccard {np.mean(jacc):.3f}")

# ── Finding precision / recall vs PTB-XL ground truth ──
rows = []
for i, s in enumerate(SUPERCLASSES):
    tp = fp = fn = 0
    for rec in test_records:
        probs = PROBS_BY_RID[rec["rid"]]
        pred = probs[i] >= THRESHOLDS[s]
        gold = s in rec["supers"]
        tp += int(pred and gold)
        fp += int(pred and not gold)
        fn += int(not pred and gold)
    rows.append({"finding": s, "precision": tp / max(tp + fp, 1),
                 "recall": tp / max(tp + fn, 1), "support": tp + fn})
pr_df = pd.DataFrame(rows).set_index("finding").round(3)
display(pr_df)

In [ ]:
# ── Narrative component: guardrail results + clinician review rubric ──
print("Numeric guardrail (per sampled case):")
for rid, bad in GUARD.items():
    status = "clean" if not bad else f"{len(bad)} violations: {', '.join(bad)}"
    print(f"  {rid}: {status} ({len(NARRATIVES[rid].split())} words)")

rubric = pd.DataFrame([
    {"criterion": "Factual consistency", "question": "Does every statement trace back to the case JSON?", "score (1-5)": ""},
    {"criterion": "No invented measurements", "question": "Are nulls reported as unavailable rather than guessed?", "score (1-5)": ""},
    {"criterion": "Diagnostic framing", "question": "Are model probabilities framed as statistical outputs, not diagnoses?", "score (1-5)": ""},
    {"criterion": "Structure", "question": "Are all five required sections present and clearly separated?", "score (1-5)": ""},
    {"criterion": "Teaching value", "question": "Do the Key Teaching Points add clarity without new claims?", "score (1-5)": ""},
    {"criterion": "Overall readability", "question": "Would a clinician find the prose clear and concise?", "score (1-5)": ""},
])
print("\nClinician review rubric (manual expert scoring, per spec - not BLEU/ROUGE):")
display(rubric)

first_rid = list(CASES.keys())[0]
print(f"\n----- full narrative example (record {first_rid}) -----")
print(NARRATIVES[first_rid])

## Summary — the constraint this notebook enforces

```
ECG ──► [1] 1D ResNet (5 sigmoid outputs)  ──┐
ECG ──► [2] signal processing + annotations ─┼──► [3] structured case JSON ──► [4] LLM phrasing ──► [5] PDF
                                              │         (validated)              (numeric guardrail)
```

The LLM never sees a waveform and never originates a number or diagnosis — it only phrases the Stage 3 JSON, and a post-hoc validator rejects any number that does not exist in that JSON.

**Artifacts written to the output directory** (Kaggle: `/kaggle/working`):

| File | Content |
|---|---|
| `resnet1d_multilabel_best.pth` | Best-validation-checkpoint model weights |
| `training_curves.png` | Loss + validation macro-AUROC curves |
| `thresholds.json` | Validation-tuned per-class decision thresholds |
| `structured_cases.json` | The Stage 3 case objects (the LLM's only input) |
| `narratives.json` | Generated narratives per sample case |
| `case_<rid>.pdf` | One-page clinical PDF reports |